# **# مهمه تركي (Data & Benchmarking)**

In [ ]:
import torch

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Install libraries required to access and analyse the AMI dataset
2
# datasets is used to load datasets from Hugging Face
3
# pandas will be used later for data analysis and dataset splitting
!pip install datasets huggingface_hub pandas




In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("edinburghcstr/ami")
print(configs)

In [ ]:
from datasets import load_dataset

metadata = load_dataset(
    "edinburghcstr/ami",
    "ihm",
    split="train",
    streaming=False
).remove_columns(["audio"])


In [ ]:
unique_speakers = set()

for sample in metadata:
    unique_speakers.add(sample["speaker_id"])

print("Unique Speakers:", len(unique_speakers))

In [ ]:
# Calculate speaker-based split sizes

total_speakers = len(unique_speakers)

train_speakers = round(total_speakers * 0.70)
validation_speakers = round(total_speakers * 0.15)

test_speakers = total_speakers - train_speakers - validation_speakers

print(f"Total Speakers: {total_speakers}")
print(f"Train Speakers (70%): {train_speakers}")
print(f"Validation Speakers (15%): {validation_speakers}")
print(f"Test Speakers (15%): {test_speakers}")

In [ ]:
# Convert unique speakers into a list
# Shuffle speakers randomly before splitting
# This helps create unbiased train/validation/test sets

import random

speaker_list = list(unique_speakers)

random.seed(42)

random.shuffle(speaker_list)

In [ ]:
# Create speaker-disjoint train, validation, and test speaker groups
# Each speaker will belong to only one dataset split

train_speaker_ids = speaker_list[:109]

validation_speaker_ids = speaker_list[109:132]

test_speaker_ids = speaker_list[132:]

print("Train Speakers:", len(train_speaker_ids))
print("Validation Speakers:", len(validation_speaker_ids))
print("Test Speakers:", len(test_speaker_ids))

In [ ]:
# Verify that no speaker appears in more than one split
# This confirms that Speaker-Disjoint Split was successfully applied

print("Train ∩ Validation =", len(set(train_speaker_ids) & set(validation_speaker_ids)))

print("Train ∩ Test =", len(set(train_speaker_ids) & set(test_speaker_ids)))

print("Validation ∩ Test =", len(set(validation_speaker_ids) & set(test_speaker_ids)))

In [ ]:
from datasets import load_dataset, Audio

# نعيد تحميل الداتاست، هذه المرة بدون حذف عمود audio
# لأن هذه المرحلة تحتاج الصوت الفعلي للمعالجة
audio_dataset = load_dataset(
    "edinburghcstr/ami",
    "ihm",
    split="train",
    streaming=True
)

# تحويل معدل العينات إلى 16kHz (lazy — يصير فقط وقت سحب العينة فعليًا)
audio_dataset = audio_dataset.cast_column("audio", Audio(sampling_rate=16000))

# نتحقق من أول عينة: معدل العينات + شكل القناة الصوتية
sample = next(iter(audio_dataset))
print("Sampling rate:", sample["audio"]["sampling_rate"])
print("Array shape:", sample["audio"]["array"].shape)

In [ ]:
# Load one sample from the ICSI dataset
# The dataset only provides a test split

from datasets import load_dataset

icsi_dataset = load_dataset(
    "argmaxinc/icsi-meetings",
    split="test",
    streaming=True
)

sample = next(iter(icsi_dataset))

print(sample)

In [ ]:
# Inspect ICSI dataset structure
# This helps identify available fields for benchmarking and evaluation

print(sample.keys())

In [ ]:
# ==========================================
# Part 1: validate a single sample
# ==========================================
# The audio column is a dict ({"array", "sampling_rate"}) on a downloaded
# dataset and a decoder object (.metadata) on a streaming one. Both shapes
# reach this function, so it reads either — otherwise every sample is
# rejected downstream and the DER loop reports zero meetings.

def audio_info(sample):
    """Return (duration_seconds, sample_rate, num_channels) for either shape."""
    audio = sample["audio"]

    if isinstance(audio, dict):
        array = audio["array"]
        rate = audio["sampling_rate"]
        channels = 1 if getattr(array, "ndim", 1) == 1 else array.shape[0]
        return len(array) / rate, rate, channels

    meta = audio.metadata
    return meta.duration_seconds, meta.sample_rate, meta.num_channels


def is_valid_sample(sample):
    # Audio must be readable and have a real duration
    try:
        duration, _rate, _channels = audio_info(sample)
        if duration <= 0:
            return False
    except Exception:
        return False

    # Speaker labels and timestamps must exist
    if len(sample["speakers"]) == 0:
        return False
    if len(sample["timestamps_start"]) == 0 or len(sample["timestamps_end"]) == 0:
        return False

    # ...and the three lists must line up
    if not (len(sample["speakers"]) == len(sample["timestamps_start"]) == len(sample["timestamps_end"])):
        return False

    return True


# ==========================================
# Part 1b: check 16kHz + Mono compliance across the dataset
# ==========================================

non_compliant = 0
checked = 0

for sample in icsi_dataset:
    checked += 1
    _duration, rate, channels = audio_info(sample)
    if rate != 16000 or channels != 1:
        non_compliant += 1

print(f"Checked: {checked}")
print(f"Not already 16kHz Mono: {non_compliant}")

::# **# مهمه تغريد (ASR & Performance)**

> Add blockquote









In [ ]:
!pip install -q openai-whisper
!pip install -q openai-whisper jiwer librosa datasets

In [ ]:
from datasets import load_dataset
import whisper
import librosa
from jiwer import wer
import time, re

# ================================
# دالة تطبيع النص قبل المقارنة
# ================================
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ================================
# دالة لاستبعاد العينات القصيرة/غير الصالحة
# ================================
def is_degenerate_sample(ref_text, min_words=3):
    return len(ref_text.split()) < min_words

# ================================
# تحميل داتا AMI + تحميل الموديل مرة واحدة فقط
# ================================
ami = load_dataset("edinburghcstr/ami", "ihm", split="test")
whisper_model = whisper.load_model("small")

def whisper_asr(audio_array, sample_rate):
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
    result = whisper_model.transcribe(audio_array, language="en")
    return result["text"]

def measure_performance(audio_array, sample_rate):
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
    start = time.time()
    result = whisper_model.transcribe(audio_array, language="en")
    end = time.time()
    return end - start, result["text"]

# ================================
# التقييم على عدة عينات (مو عينة وحدة)
# ================================
wer_scores = []
latencies = []
skipped = 0

for i, sample in enumerate(ami.select(range(100))):
    ref_raw = sample["text"]

    if is_degenerate_sample(ref_raw):
        skipped += 1
        continue

    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    pred_raw = whisper_asr(audio, sr)
    ref = normalize_text(ref_raw)
    pred = normalize_text(pred_raw)

    sample_wer = wer(ref, pred)
    wer_scores.append(sample_wer)

    latency, _ = measure_performance(audio, sr)
    latencies.append(latency)

    print(f"[{i}] REF: {ref[:50]!r} | PRED: {pred[:50]!r} | WER: {sample_wer:.2f}")

avg_wer = sum(wer_scores) / len(wer_scores) if wer_scores else None
avg_latency = sum(latencies) / len(latencies) if latencies else None

print(f"\nتم تجاهل {skipped} عينة قصيرة/غير صالحة")
print(f"متوسط WER عبر {len(wer_scores)} عينة: {avg_wer:.3f}" if avg_wer else "لا عينات صالحة")
print(f"متوسط زمن الاستجابة: {avg_latency:.3f} ثانية" if avg_latency else "")

In [ ]:
# ================================
# حساب Accuracy بشكل صحيح (بدون أرقام سالبة غريبة)
# ================================

if avg_wer is not None:
    # نمنع القيمة السالبة إذا WER تجاوز 100% لأي عينة شاذة أفلتت من الفلترة
    accuracy_percent = max(0, (1 - avg_wer) * 100)
    print(f"Whisper Accuracy (%): {accuracy_percent:.2f}")
else:
    print("لا توجد عينات كافية لحساب Accuracy")

In [ ]:
from datasets import load_dataset
import time

def load_icsi_dataset(max_retries=5, wait_seconds=2):
    """
    تحميل ICSI مع إعادة محاولات تلقائية في حال ظهور خطأ 503.
    """
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Loading ICSI dataset... Attempt {attempt}/{max_retries}")
            dataset = load_dataset("argmaxinc/icsi-meetings", split="test")
            print("ICSI dataset loaded successfully.")
            return dataset

        except Exception as e:
            print(f"Error: {e}")
            print(f"Retrying in {wait_seconds} seconds...")
            time.sleep(wait_seconds)

    print("Failed to load ICSI dataset after multiple attempts.")
    return None

# تحميل الداتا
icsi_dataset = load_icsi_dataset()

if icsi_dataset is None:
    print("Dataset could not be loaded. Please try again later.")
else:
    print("Dataset is ready for processing.")


In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install librosa

import whisper
import librosa

# تحميل نموذج Whisper
model = whisper.load_model("base")


In [ ]:
import whisper
import librosa

# تحميل نموذج Whisper (Baseline لاحقاً)
model = whisper.load_model("base")

def whisper_asr(audio_array, sample_rate):
    """
    تشغيل Whisper ASR على عينة صوتية واحدة.
    """
    # Whisper يعمل على 16kHz فقط
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)

    result = model.transcribe(audio_array)
    return result["text"]


In [ ]:
# استخراج أول عينة من ICSI
sample = next(iter(icsi_dataset))

# التأكد أن العينة تحتوي صوت
if "audio" not in sample or sample["audio"] is None:
    print("This sample has no audio.")
else:
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]


In [ ]:
pred = whisper_asr(audio, sr)
print(pred)


In [ ]:
# اختيار نموذج Whisper الأساسي (Baseline)
baseline_model = whisper.load_model("base")

def baseline_asr(audio_array, sample_rate):
    """
    تشغيل نموذج Whisper الأساسي (Baseline).
    """
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)

    result = baseline_model.transcribe(audio_array)
    return result["text"]


In [ ]:
baseline_pred = baseline_asr(audio, sr)
print(baseline_pred)


In [ ]:
from jiwer import wer

def compute_wer(reference_text, predicted_text):
    """
    حساب معدل الخطأ في الكلمات WER.
    """
    references = [reference_text]
    predictions = [predicted_text]

    if len(references) == 0 or len(predictions) == 0:
        print("No valid samples to evaluate WER.")
        return None

    score = wer(references, predictions)
    return score


In [ ]:
from datasets import load_dataset

ami = load_dataset("edinburghcstr/ami", "ihm", split="test")

sample = ami[0]

audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]
ref = sample["text"]   # النص موجود هنا

pred = whisper_asr(audio, sr)

wer_score = compute_wer(ref, pred)
print("WER:", wer_score)


In [ ]:
from datasets import load_dataset

# تحميل داتا AMI
ami = load_dataset("edinburghcstr/ami", "ihm", split="test")

# أخذ أول عينة
sample = ami[0]

# استخراج النص فقط
ref = sample["text"]

print("Transcript:", ref)


In [ ]:
from datasets import load_dataset

ami = load_dataset("edinburghcstr/ami", "ihm", split="test")

sample = ami[0]
audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]
ref = sample["text"]   # هنا النص الحقيقي

pred = whisper_asr(audio, sr)
wer_score = compute_wer(ref, pred)

print("Whisper Output:", pred)
print("WER:", wer_score)


In [ ]:
# استخراج أول عينة من ICSI
sample = next(iter(icsi_dataset))

# استخراج الصوت فقط (لأن ICSI ما فيها نصوص)
audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

# تشغيل Whisper على الصوت
pred = whisper_asr(audio, sr)
print("Whisper Output:", pred)


In [ ]:
sample = next(iter(icsi_dataset))

audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

pred = whisper_asr(audio, sr)
print("Whisper Output:", pred)


In [ ]:
# استخراج أول عينة بشكل آمن
sample = next(iter(icsi_dataset))

# التأكد أن العينة تحتوي نص
if "transcript" not in sample or sample["transcript"] is None:
    print("This sample has no transcript.")
else:
    ref = sample["transcript"]          # النص الصحيح
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    pred = whisper_asr(audio, sr)       # تشغيل Whisper

    # الآن نحسب WER بدون خطأ
    wer_score = compute_wer(ref, pred)
    print("WER:", wer_score)


In [ ]:
wer_score = compute_wer(ref, pred)
print("WER:", wer_score)


In [ ]:
import time

def measure_performance(audio_array, sample_rate):
    """
    قياس زمن المعالجة (Latency) ووقت التشغيل.
    """
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)

    start = time.time()
    result = model.transcribe(audio_array)
    end = time.time()

    latency = end - start
    return latency, result["text"]


In [ ]:
latency, output = measure_performance(audio, sr)
print("Latency:", latency)
print("Output:", output)


# ريما

In [ ]:
!pip install pyannote.audio torch soundfile

import torch
import numpy as np
from pyannote.audio import Pipeline
from pyannote.core import Segment, Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading Diarization Pipeline on: {device}")

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)
pipeline.to(device)

In [ ]:
def build_reference_annotation(sample):
   #Convert reference speaker IDs and start/end timestamps into a pyannote Annotation object.
    ref_annotation = Annotation(uri="meeting_audio")
    speakers = sample.get("speakers", [])
    starts = sample.get("timestamps_start", [])
    ends = sample.get("timestamps_end", [])

    for spk, start, end in zip(speakers, starts, ends):
        if end > start:
            ref_annotation[Segment(start, end)] = str(spk)

    return ref_annotation

In [ ]:
def run_diarization_on_sample(audio_array, sample_rate):

   # Send the audio waveform array to the PyAnnote pipeline and extract speaker segments.
    if isinstance(audio_array, np.ndarray):
        audio_tensor = torch.from_numpy(audio_array).float()
    else:
        audio_tensor = audio_array.float()

    if audio_tensor.ndim == 1:
        audio_tensor = audio_tensor.unsqueeze(0)

    waveform = {"waveform": audio_tensor, "sample_rate": sample_rate}

    # Run speaker diarization inference
    result = pipeline(waveform)

    # If pyannote returns a DiarizeOutput object, extract the Annotation
    if hasattr(result, "speaker_diarization"):
        return result.speaker_diarization

    return result

In [ ]:
#Run DER evaluation across the dataset samples

der_metric = DiarizationErrorRate()
processed_count = 0

print("\nStarting Diarization & DER Evaluation...")
print("=" * 60)

for sample in icsi_dataset:
    if not is_valid_sample(sample):
        continue

    processed_count += 1

    audio_arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Construct Ground Truth annotation
    ref_annot = build_reference_annotation(sample)

    # Run model hypothesis
    hyp_annot = run_diarization_on_sample(audio_arr, sr)

    # Compute DER for the current sample and accumulate in metric
    sample_der = der_metric(ref_annot, hyp_annot)

    print(f"Sample {processed_count}:")
    print(f"  Ground Truth Segments: {len(ref_annot)}")
    print(f"  Predicted Segments:    {len(hyp_annot)}")
    print(f"  Sample DER:            {sample_der * 100:.2f}%")

    # Display the first three detected speaker turns for inspection
    print("  First Predicted Turns:")
    for turn, _, speaker in list(hyp_annot.itertracks(yield_label=True))[:3]:
        print(f"    - [{turn.start:.2f}s -> {turn.end:.2f}s] Speaker {speaker}")
    print("-" * 60)

    # Limit to 5 samples for the initial benchmark run to control GPU runtime
    if processed_count >= 5:
        break

In [ ]:
final_der = abs(der_metric)
print("\n" + "=" * 40)
print(f"Total Meetings Evaluated: {processed_count}")
print(f"Overall Cumulative DER:   {final_der * 100:.2f}%")
print("=" * 40)

# الجوهرة

# **# Integration & Privacy**

الطبقة اللي تجمع المكوّنات الثلاثة في نظام واحد: واجهة، مسار رفع، خدمة FastAPI،
ودمج مخرجات ASR و Diarization و LLM، مع ضوابط الخصوصية الحاكمة لكل ذلك.

| المطلوب | أين يقع في الكود |
|---|---|
| UI | `app/web/` — واجهة بدون build step ولا مكتبات خارجية |
| Upload Flow | `POST /v1/jobs` — تحقّق ثم job id، والمعالجة خلف الطلب |
| FastAPI Backend | `app/main.py` + `app/api/` |
| Integration | `app/pipeline/align_core.py` — إسناد كل كلمة لمتحدثها |
| Privacy Requirements | `app/core/crypto.py` · `store.py` · `audit.py` · `retention.py` · `app/pipeline/redaction.py` |

الفكرة المعمارية: Whisper يجاوب **ماذا قيل ومتى**، و pyannote يجاوب **من تكلّم ومتى**،
ولا أحد منهما يجاوب **من قال ماذا** — وهذا هو المنتج. لذلك يُكتب هنا، في طبقة الدمج،
لا داخل أي من النموذجين. ثم كل نقطة في الملخّص تُتحقَّق مقابل النص قبل عرضها.

## 1 · إحضار الخدمة وتشغيل اختباراتها

الاختبارات تعمل على backends مكتوبة (mock) بنفس عقود النماذج الحقيقية: بلا أوزان،
بلا شبكة، فتُعطي نفس النتيجة على أي جهاز. تشغيلها أولًا هو الدليل قبل العرض.

In [ ]:
!git clone --depth 1 https://github.com/imzezsv-dot/vocalyze.git vocalyze 2>/dev/null || echo "already cloned"
%cd vocalyze
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

In [ ]:
!pytest

---

# ⬇️ فرّغ تسجيلك أنت

**قبل التشغيل:** من القائمة أعلى → `Runtime` → `Change runtime type` → اختر **T4 GPU**.

شغّل الخلية التالية، وارفع ملفك لما يطلب منك. لا تحتاج أي حساب.

*(فصل المتحدثين الحقيقي يحتاج توكن Hugging Face — إن تركته فاضيًا تحصل على تفريغ
صوتك كاملًا مع فصل تقريبي للمتحدثين، وهذا يكفي للعرض.)*

In [ ]:
!pip install -q faster-whisper pyannote.audio

import os, sys, importlib, time
sys.path.insert(0, os.getcwd())   # %cd لا يضيف المجلد الجديد إلى مسار الاستيراد

os.environ["ASR_BACKEND"]         = "whisper"       # Whisper الحقيقي
os.environ["WHISPER_MODEL"]       = "small"         # large-v3 أدق وأبطأ
os.environ["DIARIZATION_BACKEND"] = "pyannote"
os.environ["SUMMARIZER_BACKEND"]  = "extractive"
os.environ["HUGGINGFACE_TOKEN"]   = ""              # ← اتركه فاضيًا، أو ضع توكنك

if not os.environ["HUGGINGFACE_TOKEN"]:
    os.environ["DIARIZATION_BACKEND"] = "mock"
    print("بلا توكن: تفريغ صوتك حقيقي، وفصل المتحدثين تقريبي.\n")

# إعادة بناء الخدمة بالإعدادات الجديدة
import app.config as config; config.get_settings.cache_clear()
from app.pipeline import registry; registry._cache.clear()
import app.main as main; importlib.reload(main)

from fastapi.testclient import TestClient
from google.colab import files

print("ارفع ملفك الصوتي أو المرئي:")
uploaded = files.upload()
name = list(uploaded)[0]

with TestClient(main.app) as service:
    caps = service.get("/v1/capabilities").json()
    print(f"\nيعمل بـ: asr={caps['asr_backend']} diarization={caps['diarization_backend']}")

    accepted = service.post(
        "/v1/jobs",
        files={"file": (name, uploaded[name], "application/octet-stream")},
        data={"consent": "true"},
    ).json()
    job_id, token = accepted["job_id"], accepted["access_token"]
    headers = {"Authorization": f"Bearer {token}"}

    print("يشتغل الآن — أول مرة ينزّل الأوزان فيكون أبطأ...")
    while True:
        status = service.get(f"/v1/jobs/{job_id}", headers=headers).json()
        if status["state"] in ("completed", "failed"):
            break
        time.sleep(2)

    if status["state"] == "failed":
        print("\nفشل:", status["error"])
    else:
        result = service.get(f"/v1/jobs/{job_id}/result", headers=headers).json()

        print("\n" + "=" * 72)
        print("المتحدثون:", result["transcript"]["speakers"])
        print("=" * 72)
        for u in result["transcript"]["utterances"]:
            print(f"[{u['start']:7.1f}s] {u['speaker']:<12} {u['text']}")

        print("\n" + "=" * 72)
        print("الملخّص:", result["brief"]["summary"])
        print("=" * 72)
        for key, label in [("decisions", "قرار"), ("action_items", "مهمة"), ("key_points", "نقطة")]:
            for item in result["brief"][key]:
                print(f"• [{label}] {item['text']}")
                print(f"          الدليل: {item['evidence']['utterance_ids']}")

---

# 🔗 موقع حي يفرّغ الصوت — بلا اشتراك

الخلية التالية تشغّل الخدمة كاملة داخل Colab، وتفتح لها **رابطًا عامًا** يقدر أي
أحد يفتحه من جواله أو حاسبه: يرفع تسجيلًا، ويستقبل نصًا منسوبًا وملخّصًا مُتحقَّقًا —
بـ Whisper حقيقي على معالج الرسوميات المجاني.

الرابط يعيش ما دام الدفتر شغّالًا. للعرض المباشر هذا يكفي؛ ولو أردت رابطًا دائمًا
فذلك يحتاج استضافة مدفوعة.

In [ ]:
# الخدمة كاملة + رابط عام. لا يحتاج أي حساب.
!pip install -q faster-whisper
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os, sys, re, time, subprocess
sys.path.insert(0, os.getcwd())

os.environ.update(
    ASR_BACKEND="whisper",            # Whisper الحقيقي
    WHISPER_MODEL="small",
    DIARIZATION_BACKEND="mock",       # pyannote يحتاج توكن — انظر الخلية السابقة
    SUMMARIZER_BACKEND="extractive",
    DATA_DIR="/content/vocalyze-data",
    MAX_UPLOAD_MB="200",
)
from app.core.crypto import generate_service_key
os.environ["ENCRYPTION_KEY"] = generate_service_key()

# تشغيل الخدمة
service = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("تشغيل الخدمة...")
time.sleep(10)

# فتح رابط عام
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url = None
for line in tunnel.stdout:
    found = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if found:
        url = found.group(0)
        break

if url:
    print("\n" + "=" * 60)
    print("🔗 موقعك الحي:", url)
    print("=" * 60)
    print("افتحه، ارفع تسجيلك، واحصل على النص. يعمل ما دام هذا الدفتر شغّالًا.")
else:
    print("تعذّر فتح النفق. أعد تشغيل الخلية.")

---

## 2 · تشغيل الخدمة داخل الدفتر

`TestClient` يشغّل تطبيق FastAPI نفسه داخل هذه العملية — نفس الكود الذي يعمل على
الخادم، بلا منفذ شبكة. مفتاح التشفير يُولَّد الآن ويُحفظ في البيئة فقط.

In [ ]:
import os, sys, time, json
sys.path.insert(0, os.getcwd())

from app.core.crypto import generate_service_key

os.environ["ENCRYPTION_KEY"] = generate_service_key()   # مفتاح الخدمة، خارج المستودع
os.environ["DATA_DIR"] = "/content/vocalyze-data"
os.environ["ASR_BACKEND"] = "mock"                      # بدّلها إلى whisper مع النموذج الحقيقي
os.environ["DIARIZATION_BACKEND"] = "mock"              # أو pyannote
os.environ["SUMMARIZER_BACKEND"] = "mock"               # أو llm

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()      # يشغّل دورة حياة الخدمة: العامل، ومكنسة الاحتفاظ

print(json.dumps(client.get("/v1/capabilities").json(), indent=2))

`demo_mode: true` معناها أن هذه النسخة تعمل بنماذج مكتوبة. الواجهة تقرأ هذا الحقل
وتُعلن عن نفسها به — عرض عيّنة مكتوبة على أنها اجتماع المستخدم كذب، لا ميزة.

## 3 · مسار الرفع: من ملف صوتي إلى نص منسوب

الرفع تبادل من خطوتين لا طلب واحد طويل: يُتحقَّق من الملف ويُقبل، والمعالجة تجري خلف
`job id`. اجتماع تسعين دقيقة يستغرق دقائق في التفريغ، وطلب POST مفتوح طوال تلك المدة
يسقط عند أول انقطاع أو مهلة وسيط.

In [ ]:
# ملف صوتي حقيقي (نغمة) — يكفي لإثبات مسار الترميز والتطبيع إلى 16kHz mono
!ffmpeg -y -f lavfi -i "sine=frequency=220:duration=12" -ar 44100 -ac 2 /content/meeting.wav -loglevel error

# 1) محاولة رفع بلا موافقة مسجّلة: مرفوضة قبل أن تُلمس البايتات
refused = client.post(
    "/v1/jobs",
    files={"file": ("meeting.wav", open("/content/meeting.wav", "rb"), "audio/wav")},
    data={"consent": "false"},
)
print(refused.status_code, refused.json()["error"], "→", refused.json()["fix"])

In [ ]:
# 2) الرفع الصحيح
accepted = client.post(
    "/v1/jobs",
    files={"file": ("meeting.wav", open("/content/meeting.wav", "rb"), "audio/wav")},
    data={"consent": "true"},
).json()

job_id  = accepted["job_id"]
token   = accepted["access_token"]          # الـ id وحده لا يمنح شيئًا
headers = {"Authorization": f"Bearer {token}"}

print("job:", job_id, "| state:", accepted["state"], "| expires:", accepted["expires_at"])

# 3) متابعة المراحل الخمس حتى الاكتمال
while True:
    job = client.get(f"/v1/jobs/{job_id}", headers=headers).json()
    if job["state"] in ("completed", "failed"):
        break
    time.sleep(0.3)

for stage in job["stages"]:
    print(f"  {stage['name']:<13} {stage['state']:<8} {stage.get('detail') or ''}")
print("\nstate:", job["state"], "| error:", job["error"])

## 4 · الدمج: من قال ماذا

ساعتا النموذجين لا تتطابقان — توقيتات Whisper تنزلق عند حواف المقاطع، وحدود أدوار
pyannote تقع وسط الكلمة، وكلاهما يخطئ أثناء التداخل. لذلك الإسناد قائم على **تعظيم
التداخل** لا مطابقة الحدود: كل كلمة تذهب للمتحدث الذي تغطي أدواره أكبر جزء منها،
ومع وجود مخرج صريح `UNKNOWN` بدل التخمين حين لا يوجد تداخل أصلًا.

In [ ]:
result = client.get(f"/v1/jobs/{job_id}/result", headers=headers).json()
transcript = result["transcript"]

print(f"المتحدثون: {transcript['speakers']}")
print(f"المدة: {transcript['duration']}s | عدد الأسطر: {len(transcript['utterances'])}\n")

for u in transcript["utterances"][:8]:
    flags = []
    if u["overlapped"]:
        flags.append("تداخل")
    if u.get("redacted"):
        flags.append("حُذف معرّف")
    mark = f"  [{'، '.join(flags)}]" if flags else ""
    print(f"[{u['id']:>3}] {u['speaker']:<12} {u['start']:>6.1f}s  {u['text'][:70]}{mark}")

## 5 · منع الهلوسة: كل نقطة تحمل السطر الذي جاءت منه

النموذج اللغوي حين يُطلب منه تلخيص اجتماع يُنتج أحيانًا قرارًا لم يتخذه أحد. الـ prompt
يقلّل ذلك ولا يُنهيه. لذلك تُعامل كل نقطة مولَّدة هنا كدعوى تُتحقَّق مقابل النص:

1. لا بد أن تستشهد بمعرّفات أسطر، وأن تكون تلك الأسطر موجودة فعلًا؛
2. لا بد أن تظهر كلمات الدعوى المضمونية في الأسطر المستشهد بها؛
3. كل رقم في الدعوى لا بد أن يظهر في الدليل — الأرقام المخترعة أشد ضررًا وأسهل كشفًا.

ما لا يجتاز ذلك يُحذف ويُعدّ، والعدد معروض في الواجهة.

In [ ]:
brief   = result["brief"]
quality = result["quality"]
by_id   = {u["id"]: u["text"] for u in transcript["utterances"]}

print("الملخّص:", brief["summary"], "\n")

for label, key in [("القرارات", "decisions"), ("المهام", "action_items"), ("النقاط", "key_points")]:
    if not brief[key]:
        continue
    print(f"— {label} —")
    for item in brief[key]:
        evidence = item["evidence"]
        owner = f" (المسؤول: {item['owner']})" if item.get("owner") else ""
        print(f"  • {item['text']}{owner}")
        print(f"    الدليل {evidence['utterance_ids']} — تطابق {evidence['grounding']:.0%}")
        for uid in evidence["utterance_ids"]:
            print(f"      {uid}: {by_id.get(uid, '')[:70]}")
    print()

print(f"نقاط مُتحقَّق منها: {quality['grounded_claims']} | نقاط حُذفت لعدم وجود سند: {quality['dropped_claims']}")

### التحقّق كوحدة قائمة بذاتها

`grounding_core` لا يعتمد على FastAPI ولا على النماذج، فيمكن اختباره وحده. هنا دعوى
صادقة ودعوى مطابقة لها لفظًا إلا في الرقم — وهذا بالضبط ما ينبغي أن يسقط.

In [ ]:
from app.pipeline.grounding_core import Retriever, ground_claim

lines = [
    {"id": "u1", "text": "Word error rate is eleven point two percent on the clean split."},
    {"id": "u2", "text": "We agreed to flag overlapping speech instead of forcing a single speaker."},
]
by_uid    = {u["id"]: u for u in lines}
retriever = Retriever(lines)

for claim, cited in [
    ("Word error rate is eleven point two percent on the clean split", ["u1"]),  # صحيحة
    ("Word error rate is 47 percent on the clean split",               ["u1"]),  # رقم مخترع
    ("The team decided to cancel the project",                         ["u2"]),  # لم تُقل
    ("Overlapping speech is flagged instead of forcing one speaker",    None),   # صحيحة بلا استشهاد
]:
    verdict = ground_claim(claim, cited, by_uid, retriever)
    status  = "مقبولة" if verdict.verified else "مرفوضة"
    print(f"{status:<7} ({verdict.score:.2f}) {claim[:56]}")
    if verdict.reason:
        print(f"         السبب: {verdict.reason}")

## 6 · متطلبات الخصوصية — بوصفها تأكيدات لا وعودًا

الخصوصية مفروضة **بين** المراحل لا عند الأطراف، لأن هناك تتحرك البيانات فعلًا: الصوت
الخام يُشفَّر لحظة وصوله بمفتاح خاص بهذه المهمة وحدها، ويُتلف فور قراءته، والمعرّفات
تُحذف قبل التخزين وقبل أن يصل أي نص إلى المُلخِّص.

In [ ]:
policy = client.get("/v1/privacy/policy").json()
print(json.dumps(policy, indent=2, ensure_ascii=False)[:900], "…\n")

privacy = result["privacy"]
print("الصوت ما زال مخزَّنًا؟ ", privacy["audio_retained"])
print("مشفَّر على القرص؟     ", privacy["encrypted_at_rest"])
print("موافقة مسجَّلة؟       ", privacy["consent"])
print("مدة الاحتفاظ (ساعة):  ", privacy["retention_hours"])
print("النماذج المستخدمة:    ", privacy["models"])

In [ ]:
# البايتات على القرص فعلًا غير قابلة للقراءة
from pathlib import Path

blob = Path(os.environ["DATA_DIR"]) / "jobs" / job_id / "result.enc"
raw  = blob.read_bytes()
print("حجم الملف المشفَّر:", len(raw), "بايت")
print("أول 48 بايت:", raw[:48])

first_line = transcript["utterances"][0]["text"]
print("\nهل يظهر نص الاجتماع داخل الملف؟", first_line.encode() in raw)

In [ ]:
# سجل التدقيق: ماذا فعل النظام بهذا التسجيل — أفعال فقط، لا محتوى
trail = client.get(f"/v1/privacy/jobs/{job_id}/audit", headers=headers).json()
for entry in trail["entries"]:
    print(f"{entry['time'][11:19]}  {entry['action']:<22} {entry['details']}")

# وكل سطر يحمل بصمة السطر الذي قبله، فحذف أو تعديل واحد يكسر السلسلة
print("\nسلامة السلسلة:", client.get("/v1/privacy/audit/verify").json())

In [ ]:
# حذف المعرّفات: ضيّق عمدًا — الميزانيات والتواريخ والنسب هي مادة الاجتماع
from app.pipeline.redaction import redact_text

for line in [
    "راسلني على layla.ahmed@example.com أو اتصل +966 50 123 4567",
    "معدل الخطأ 11.2 بالمئة والميزانية 250000 ريال والإصدار 3.11.9",
]:
    cleaned, report = redact_text(line)
    print(f"قبل : {line}")
    print(f"بعد : {cleaned}")
    print(f"       حُذف {report.count} ({report.by_kind})\n")

In [ ]:
# الحق في المحو: تُتلف الكتل ويُدمَّر مفتاح المهمة، فأي نسخة باقية تبقى شفرة
print(client.delete(f"/v1/jobs/{job_id}", headers=headers).json()["message"])

after = client.get(f"/v1/jobs/{job_id}", headers=headers)
print("\nقراءة بعد الحذف:", after.status_code, after.json()["error"])
print("هل بقيت أي كتلة على القرص؟",
      list((Path(os.environ["DATA_DIR"]) / "jobs" / job_id).glob("*.enc")))

## 7 · كيف تدخل نماذج الفريق الحقيقية

طبقة الدمج لا تستورد Whisper ولا pyannote مباشرة. تعتمد على ثلاثة عقود في
`app/pipeline/` — `ASRBackend` و `DiarizationBackend` و `SummarizerBackend` — فيستطيع
صاحب كل مكوّن تبديل تنفيذه دون أن يتغيّر سطر في الـ API أو الـ aligner أو الواجهة.
التبديل بمتغيّر بيئة، لا بتعديل كود:

```ini
ASR_BACKEND=whisper                       # مكوّن التعرّف على الكلام
DIARIZATION_BACKEND=pyannote              # مكوّن فصل المتحدثين
HUGGINGFACE_TOKEN=hf_…                    # بعد قبول رخصة pyannote
SUMMARIZER_BACKEND=llm                    # مكوّن التلخيص
LLM_BASE_URL=http://127.0.0.1:11434/v1    # نموذج محلي
```

In [ ]:
import inspect
from app.pipeline.asr import ASRBackend, ASRResult
from app.pipeline.diarization import DiarizationBackend
from app.pipeline.summarizer import SummarizerBackend

for contract in (ASRBackend, DiarizationBackend, SummarizerBackend):
    print(inspect.getsource(contract))

الشرط الوحيد أن يعيد التنفيذ نفس الأشكال المعرَّفة في `app/schemas.py`. وأهمّها
`words` داخل `ASRResult`: توقيتات الكلمات هي ما يجعل نسبة الكلام إلى قائله دقيقة،
فالـ backend الذي لا ينتجها يُضعف المنتج كله — وعندها يتراجع النظام إلى تقسيم تناسبي
للمقطع بين المتحدثين، ويصرّح بذلك عبر `speaker_confidence` أقل.

In [ ]:
# مثال: تنفيذ ASR بديل — لا يُعدَّل شيء في بقية النظام
from pathlib import Path
from app.schemas import ASRResult, ASRSegment, Word

class MyASR:
    # أي تنفيذ يعيد ASRResult يعمل فورًا داخل النظام
    name = "custom"

    def __init__(self, settings=None):
        self.settings = settings

    def transcribe(self, audio_path: Path) -> ASRResult:
        return ASRResult(
            language="ar",
            duration=2.0,
            segments=[ASRSegment(
                start=0.0, end=2.0, text="مرحبا بالجميع", confidence=0.93,
                words=[Word(start=0.0, end=0.9, text="مرحبا",   confidence=0.95),
                       Word(start=1.0, end=2.0, text="بالجميع", confidence=0.91)],
            )],
            model="demo", backend="custom",
        )

# يمرّ عبر نفس aligner الإنتاج
from app.pipeline.alignment import build_transcript
from app.schemas import DiarizationResult, SpeakerTurn

turns = DiarizationResult(
    turns=[SpeakerTurn(start=0.0, end=0.95, speaker="SPEAKER_00"),
           SpeakerTurn(start=0.95, end=2.0, speaker="SPEAKER_01")],
    num_speakers=2, backend="custom", model="demo",
)
merged, _stats = build_transcript(MyASR().transcribe(Path("/dev/null")), turns)
for u in merged.utterances:
    print(f"{u.speaker}: {u.text}   (ثقة الإسناد {u.speaker_confidence})")

## 8 · الواجهة

`app/web/` — ثلاثة ملفات تخدمها نفس العملية التي تخدم الـ API: بلا build step وبلا
إطار عمل وبلا أي نداء لشبكة خارجية، فانقطاع CDN لا يُسقط الواجهة. تقرأ الواجهة
`/v1/capabilities` عند التحميل فتصف النسخة العاملة كما هي، وكل نقطة في الملخّص تحمل
رقاقة قابلة للضغط تُنزلق بالنص إلى السطر الذي جاءت منه.

للتشغيل محليًا:

```bash
./run.sh          # بيئة افتراضية + مفتاح تشفير + uvicorn، بأمر واحد
# ثم http://127.0.0.1:8000
```

In [ ]:
client.__exit__(None, None, None)   # إيقاف الخدمة وتنظيف الموارد
print("تم إغلاق الخدمة.")